In [ ]:
#| default_exp host

In [ ]:
#| hide
from fastcore.test import *

A thin app's launcher starts the server with `<PREFIX>HOST_URL` in its environment and the token as one line on stdin (`contract/api-1.md`). This module is what the server does with them. `HostClient` holds the control server's address and token, and asks the launcher what only it can do: a native question, a one-time grant, an update, an extra. It needs only the standard library and fastcore, so the server imports it with no window code.

In [ ]:
#| export
from __future__ import annotations
import http.client, json, os, select, sys
from urllib.parse import urlsplit
from fastcore.all import Path

### The names the launcher owns

`owned` lists what the launcher sets for its child. No login shell and no stored value may replace these, or the names in `STEER`, which would move the server off the env the launcher built. `host_env` is the environment block the launcher adds; the lock tests use it to start a server as the launcher does.

In [ ]:
#| export
STEER = ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE', 'VIRTUAL_ENV', 'CONDA_PREFIX')
HOST_KEYS = ('HOST_URL', 'HOST_TOKEN', 'APP_VERSION', 'APP_BUNDLE', 'LOCK_N', 'ENV_HASH', 'UV')
TOKEN_WAIT = 5

def owned(prefix): return tuple(prefix + k for k in HOST_KEYS)

def host_env(prefix, url, version='0.0.0-test', bundle='', n=0, hash='', uv=''):
    "The variables the launcher gives its server; never the token, which goes on stdin."
    return {f'{prefix}HOST_URL': url, f'{prefix}APP_VERSION': version, f'{prefix}APP_BUNDLE': bundle, f'{prefix}LOCK_N': str(n), f'{prefix}ENV_HASH': hash, f'{prefix}UV': str(uv)}

In [ ]:
test_eq(owned('LEELA_')[:2], ('LEELA_HOST_URL', 'LEELA_HOST_TOKEN'))
test_eq('LEELA_HOST_TOKEN' in host_env('LEELA_', 'http://127.0.0.1:9'), False)

### The client

`HostClient.take` is the first thing a server does in host mode. It reads the token line, keeps it in memory, and puts stdin on `/dev/null`, so no child can read the token. A token in the environment is ignored with a warning. No line, an empty line or a late line stops the server.

In [ ]:
#| export
class HostError(Exception): "The launcher did not answer, or refused."

def _read_line(fd, timeout):
    "One line from `fd`, a byte at a time so nothing past it is taken; '' at end of file or when it is late."
    out = b''
    while not out.endswith(b'\n') and len(out) < 1024:
        if not select.select([fd], [], [], timeout)[0]: return ''
        if not (b := os.read(fd, 1)): break
        out += b
    return out.decode('ascii', 'replace').strip()

class HostClient:
    "The server's side of the launcher's control server: its address, its token, and the names under `prefix`."
    def __init__(self, prefix, app='the app', data=''):
        self.prefix, self.app, self.data, self.state = prefix, app, data or app.lower(), dict(url='', token='')
    def on(self): return bool(self.state['url'])
    def is_app(self): return bool(getattr(sys, 'frozen', False)) or self.on()
    def steered(self, k): return k in STEER or k in owned(self.prefix)
    def app_version(self): return os.environ.get(f'{self.prefix}APP_VERSION') if self.on() else None
    def lock_n(self):
        try: return int(os.environ[f'{self.prefix}LOCK_N'])
        except (KeyError, ValueError): return None

    def take(self, env=None, fd=0, timeout=None):
        "Host mode: the control URL from `env`, the token as one line on `fd`, then `fd` on /dev/null. Whether host mode is on."
        env, p = os.environ if env is None else env, self.prefix
        if env.pop(f'{p}HOST_TOKEN', None) is not None: print(f'{self.app.lower()}: {p}HOST_TOKEN in the environment is ignored; the {self.app} app gives the token on stdin', file=sys.stderr)
        if not (url := env.pop(f'{p}HOST_URL', '')): return self.on()
        token = _read_line(fd, TOKEN_WAIT if timeout is None else timeout)
        null = os.open(os.devnull, os.O_RDONLY)
        os.dup2(null, fd); os.close(null)
        if not token: raise SystemExit(f'{self.app.lower()}: {p}HOST_URL is set, but no token line came on stdin. Only the {self.app} app starts the server this way.')
        self.state.update(url=url, token=token)
        return self.on()

In [ ]:
r, w = os.pipe(); os.write(w, b'tok\nrest'); os.close(w)
h, e = HostClient('DEMO_', 'Demo'), {'DEMO_HOST_URL': 'http://127.0.0.1:9', 'DEMO_HOST_TOKEN': 'x'}
test_eq((h.take(e, fd=r, timeout=1), h.state['token'], e), (True, 'tok', {}))
test_eq(os.read(r, 4), b'')
os.close(r)

`call` makes one request to the launcher. It goes only to an `http` URL on 127.0.0.1 with a port, never through a proxy. Connecting fails fast; the answer may wait for a person.

In [ ]:
#| export
def _where(self):
    u = urlsplit(self.state['url'])
    if u.scheme != 'http' or u.username or u.password or u.hostname != '127.0.0.1' or not u.port: raise HostError(f'not a loopback control server: {self.state["url"]!r}')
    return u.port

def _call(self, path, body=None, read=600, connect=3):
    "One request to the launcher: POST with a JSON `body`, else GET."
    if not self.on(): raise HostError(f'no {self.app} app runs this server')
    c = http.client.HTTPConnection('127.0.0.1', self._where(), timeout=connect)
    try:
        c.connect(); c.sock.settimeout(read)
        data = None if body is None else json.dumps(body).encode()
        c.request('GET' if body is None else 'POST', path, data, {'Authorization': f'Bearer {self.state["token"]}', 'Content-Type': 'application/json'})
        r = c.getresponse(); raw = r.read()
        if r.status != 200: raise HostError(f'the {self.app} app answered {r.status} to {path}')
        d = json.loads(raw)
        if not isinstance(d, dict): raise HostError(f'the {self.app} app gave no object for {path}')
        return d
    except HostError: raise
    except Exception as e: raise HostError(f'the {self.app} app did not answer {path}: {type(e).__name__}: {e}') from e
    finally: c.close()
HostClient._where, HostClient.call = _where, _call

In [ ]:
h = HostClient('DEMO_'); h.state['url'] = 'http://example.com:80'
test_fail(lambda: h.call('/status'), contains='not a loopback')

The control routes a server calls by name. `grant_ok` asks whether the launcher issued a one-time nonce for a path; that is how a Finder open or a picked folder is trusted, with no token in the server's hands. `ask` is a native question; any failure raises `HostError`, so a broken answer is never a No.

In [ ]:
#| export
def _grant_ok(self, nonce, path):
    "Whether the launcher issued the one-time `nonce` for `path`."
    if not (self.on() and nonce): return False
    try: return self.call('/grant', {'nonce': str(nonce), 'path': str(path)}, read=10).get('ok') is True
    except HostError: return False

def _status(self, read=5):
    "The launcher's `/status`, or {} when it does not answer."
    try: return self.call('/status', read=read)
    except HostError: return {}

def _ask(self, question):
    "A native yes or no from the launcher; HostError when no window can ask."
    d = self.call('/confirm', {'question': str(question)})
    if not isinstance(d.get('answer'), bool): raise HostError(f'the {self.app} app gave no answer')
    return d['answer']

def _health(self, version, **kw):
    "What `/health` answers: the launcher waits for `status` ok."
    return dict(status='ok', version=version, lock_n=self.lock_n()) | kw
HostClient.grant_ok, HostClient.status, HostClient.ask, HostClient.health = _grant_ok, _status, _ask, _health

In [ ]:
h = HostClient('DEMO_')
test_eq((h.grant_ok('n', '/x'), h.status()), (False, {}))
test_eq(h.health('1.2')['status'], 'ok')

### A clean environment for the server's children

Launcher 1.0.0 gave its server uv's build settings, `PYTHONNOUSERSITE`, `NO_COLOR`, `VIRTUAL_ENV` and the env's `bin` on `PATH`. Every kernel, terminal and project `uv` run inherits the server's environment, so `scrub` takes these out first. A newer lock still runs under the old launcher.

In [ ]:
#| export
def scrub(env=None, prefix=None):
    "Take out uv's build settings, PYTHONNOUSERSITE, NO_COLOR=1, a VIRTUAL_ENV that is this env, and this env's bin on PATH."
    env, prefix = os.environ if env is None else env, os.path.realpath(sys.prefix if prefix is None else prefix)
    for k in [k for k in env if k.startswith('UV_')]: del env[k]
    env.pop('PYTHONNOUSERSITE', None)
    if env.get('NO_COLOR') == '1': del env['NO_COLOR']
    if os.path.realpath(env.get('VIRTUAL_ENV') or '/') == prefix: del env['VIRTUAL_ENV']
    if 'PATH' in env: env['PATH'] = os.pathsep.join(p for p in env['PATH'].split(os.pathsep) if p and os.path.realpath(p) != os.path.join(prefix, 'bin'))
    return env

In [ ]:
e = scrub(dict(UV_X='1', PYTHONNOUSERSITE='1', NO_COLOR='1', VIRTUAL_ENV='/e', PATH='/e/bin:/usr/bin'), prefix='/e')
test_eq(e, dict(PATH='/usr/bin'))

### The app's tools on an agent's PATH

An agent's shell runs the console scripts of some packages in the app's env by name. `link_tools` links them into `bin` beside the launcher's `envs/`, so an env switch changes them. On Windows a link needs rights a person may not have, so each tool is a `.cmd` shim.

In [ ]:
#| export
def _tools_dir(self, prefix=None):
    "Where the tool links go: `bin` beside the launcher's `envs/`, else the app's data folder."
    from fastcore.xdg import xdg_data_home
    p = Path(sys.prefix if prefix is None else prefix)
    return p.parent.parent/'bin' if p.parent.name == 'envs' else xdg_data_home()/self.data/'bin'

def agent_scripts(pkgs):
    "The console scripts of `pkgs` in this env; never a python, pip or uv."
    from importlib.metadata import PackageNotFoundError, distribution
    out = []
    for n in pkgs:
        try: out += [e.name for e in distribution(n).entry_points if e.group == 'console_scripts']
        except PackageNotFoundError: pass
    return sorted(n for n in set(out) if not n.startswith(('python', 'pip', 'uv')))

def link_tools(names, src=None, dst=None, windows=None):
    "Link `names` from the scripts folder `src` into `dst`, and remove what an earlier env left there. The folder."
    import sysconfig
    win = sys.platform == 'win32' if windows is None else windows
    src, dst = Path(src or sysconfig.get_path('scripts')), Path(dst)
    exe = lambda n: src/f'{n}.exe' if win else src/n
    want = {n for n in names if exe(n).is_file()}
    dst.mkdir(parents=True, exist_ok=True)
    for n in want:
        if win: (dst/f'{n}.cmd').write_text(f'@"{exe(n)}" %*\r\n'); continue
        (tmp := dst/f'.{n}.{os.getpid()}').unlink(missing_ok=True)
        tmp.symlink_to(src/n); os.replace(tmp, dst/n)
    for p in dst.iterdir():
        if (p.stem if win and p.suffix == '.cmd' else p.name) not in want and (p.is_symlink() or (win and p.suffix == '.cmd')): p.unlink()
    return dst

def _with_tools(self, env=None, dst=None):
    "`env` with the tools folder first on PATH, in host mode."
    env = dict(os.environ if env is None else env)
    if self.on() and (d := Path(dst or self.tools_dir())).is_dir(): env['PATH'] = os.pathsep.join([str(d), *filter(None, env.get('PATH', '').split(os.pathsep))])
    return env
HostClient.tools_dir, HostClient.with_tools = _tools_dir, _with_tools

In [ ]:
import tempfile
s, d = Path(tempfile.mkdtemp()), Path(tempfile.mkdtemp())
(s/'kosha').write_text(''); (s/'kosha.exe').write_text(''); (d/'old.cmd').write_text('')
link_tools(['kosha', 'none'], s, d, windows=True)
test_eq(sorted(p.name for p in d.iterdir()), ['kosha.cmd'])
d2 = link_tools(['kosha'], s, Path(tempfile.mkdtemp()), windows=False)
test_eq((d2/'kosha').is_symlink(), True)
test_eq(HostClient('X_', 'Demo').tools_dir('/a/envs/0123'), Path('/a/bin'))